# 基于数据分析与机器学习的客户流失分析及留存决策

## 1. 项目背景

客户流失会影响企业的长期收入和客户生命周期价值，因此识别潜在流失客户以及分析影响客户流失的主要因素，对于制定客户留存策略具有重要意义。

本项目基于公开的客户行为数据构建客户流失业务场景，通过数据分析和机器学习方法识别影响客户流失的关键因素，并进一步为客户留存决策提供数据支持。

## 2. 业务问题

本项目主要回答以下问题：

**如何利用客户数据识别与客户流失相关的因素，并帮助企业制定更有效的客户留存决策？**

## 3. 项目目标

本项目将：

- 对客户数据进行质量检查和清洗
- 分析不同客户特征与流失之间的关系
- 识别具有较高流失风险的客户群体
- 建立机器学习模型预测客户流失概率
- 根据分析结果提出客户留存策略

## 4. 数据来源

本项目使用 IBM Telco Customer Churn 公开数据集进行分析。

In [4]:
from pathlib import Path
import pandas as pd
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
raw_path = project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
assert raw_path.is_file(), f"数据文件不存在！:{raw_path}"
df = pd.read_csv(raw_path)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 5. 数据初步审查（Data Audit）

在正式进行数据分析之前，首先对数据集的规模、字段结构、数据类型和数据质量进行初步检查，以确保后续分析结果的可靠性。

In [5]:
df.shape

(7043, 21)

In [6]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [8]:
df["TotalCharges"].str.strip().eq("").sum()

np.int64(11)

In [9]:
df[df["TotalCharges"].str.strip() == ""][["customerID","tenure","MonthlyCharges","TotalCharges","Churn"]]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


### TotalCharges 字段异常

通过数据类型检查发现，`TotalCharges`（累计费用）被识别为字符串类型，而从业务含义来看，该字段应属于数值型变量。

进一步检查发现，共有 11 条记录的 `TotalCharges` 为空。观察这些客户后发现，其 `tenure` 均为 0，说明这些客户可能刚加入服务，尚未形成累计费用记录。

因此，在后续数据清洗中，将首先把 `TotalCharges` 转换为数值类型，并仅针对这类 `tenure = 0` 的记录将缺失的累计费用设为 0。

In [10]:
pd.to_numeric(df["TotalCharges"], errors= "coerce")

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: TotalCharges, Length: 7043, dtype: float64

In [11]:
df["TotalCharges"]= pd.to_numeric(df["TotalCharges"], errors= "coerce")

In [12]:
df["TotalCharges"].isna().sum()

np.int64(11)

In [13]:
df.loc[(df["tenure"] == 0)&(df["TotalCharges"].isna()), "TotalCharges" ] = 0

In [14]:
df["TotalCharges"].isna().sum()

np.int64(0)

In [15]:
df["TotalCharges"].dtype

dtype('float64')

### 重复值与缺失值检查

在进一步分析之前，需要检查数据集中是否存在重复客户、重复记录以及其他缺失值，避免数据质量问题影响后续统计分析和模型训练。

In [16]:
df.duplicated().sum()

np.int64(0)

In [17]:
df["customerID"].duplicated().sum()

np.int64(0)

In [18]:
df.isna().sum().sort_values(ascending = False)

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [19]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [20]:
df["Churn"].value_counts(normalize = True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [21]:
df["Churn"].value_counts(normalize = True).mul(100).round(2)

Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64

### 数据质量检查结论

数据集中未发现重复记录或重复客户编号。在处理 `TotalCharges` 字段后，各字段均不存在缺失值。

目标变量 `Churn` 的分布如下：

- 未流失客户（No）：5174 人，占 73.46%
- 流失客户（Yes）：1869 人，占 26.54%

可以看出，数据集中未流失客户数量明显高于流失客户，存在一定程度的类别不平衡。因此，在后续模型评估中，除准确率（Accuracy）外，还需要重点关注 Recall、Precision、F1-score 和 ROC-AUC 等指标。

## 6. 字段结构与业务含义

在进行探索性分析之前，对各字段的数据类型、唯一值数量和业务含义进行进一步检查，以区分客户标识、数值变量、分类变量和目标变量。

In [22]:
field_audit = pd.DataFrame({
    "数据类型": df.dtypes,
    "唯一值数量": df.nunique(),
    "缺失值数量": df.isna().sum(),
})

field_audit

,数据类型,唯一值数量,缺失值数量
customerID,str,7043,0
gender,str,2,0
SeniorCitizen,int64,2,0
Partner,str,2,0
Dependents,str,2,0
tenure,int64,73,0
PhoneService,str,2,0
MultipleLines,str,3,0
InternetService,str,3,0
OnlineSecurity,str,3,0


In [23]:
df["Contract"].unique()

<StringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str

In [24]:
df["InternetService"].unique()

<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

In [25]:
df["PaymentMethod"].unique()

<StringArray>
[         'Electronic check',              'Mailed check',
 'Bank transfer (automatic)',   'Credit card (automatic)']
Length: 4, dtype: str

In [26]:
df["TechSupport"].unique()

<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

## 7. 数据审查总结

本次数据审查得到以下结论：

- 数据集包含 7043 条客户记录和 21 个字段
- 未发现重复记录或重复客户编号
- `TotalCharges` 原始类型为字符串，原因是其中存在 11 条空字符串记录
- 这 11 条记录的 `tenure` 均为 0，因此将其累计费用设为 0，并将 `TotalCharges` 转换为数值类型
- 数据清洗完成后不存在缺失值
- 流失客户占 26.54%，未流失客户占 73.46%，目标变量存在一定程度的类别不平衡
- 数据包含客户属性、服务使用情况、合同与支付信息以及费用信息，可用于进一步分析客户流失特征

In [27]:
output_path = project_root/ "data" / "processed" / "customer_churn_cleaned.csv"
output_path.parent.mkdir(parents = True, exist_ok = True)   
df.to_csv(output_path, index = False)
print("数据保存成功",output_path)

数据保存成功 d:\pythonProject\customer-churn-analysis\data\processed\customer_churn_cleaned.csv
